# 01 — Historical ETF data

This notebook downloads daily adjusted closing prices for the 12 ETFs
used in the market risk study. It saves a data snapshot and records
the source, requested date range, and download date.

The acquisition ends before 2018. The evaluation period is reserved
until the model is frozen.

In [20]:
%pip install -q yfinance pyarrow

In [21]:
import json
from pathlib import Path

import pandas as pd
import pyarrow
import yfinance as yf

print("pandas:", pd.__version__)
print("yfinance:", yf.__version__)
print("pyarrow:", pyarrow.__version__)

pandas: 2.2.3
yfinance: 0.2.66
pyarrow: 23.0.1


In [22]:
START_DATE = "2008-01-01"
END_DATE = "2018-01-01"

def download_daily_data(tickers):
    return yf.download(
        tickers=tickers,
        start=START_DATE,
        end=END_DATE,
        interval="1d",
        auto_adjust=False,
        group_by="column",
        multi_level_index=True,
        ignore_tz=True,
        keepna=True,
        progress=False,
    )

pilot_tickers = ["SPY", "IEF"]
pilot_data = download_daily_data(pilot_tickers)

display(pilot_data.head())

Price       Adj Close                  Close                   High  \
Ticker            IEF         SPY        IEF         SPY        IEF   
Date                                                                  
2008-01-02  55.000565  102.973946  87.629997  144.929993  87.730003   
2008-01-03  55.113503  102.924202  87.809998  144.860001  87.839996   
2008-01-04  55.257904  100.401901  88.040001  141.309998  88.239998   
2008-01-07  55.358276  100.316666  88.199997  141.190002  88.300003   
2008-01-08  55.477562   98.696739  88.389999  138.910004  88.419998   

Price                         Low                   Open              Volume  \
Ticker             SPY        IEF         SPY        IEF         SPY     IEF   
Date                                                                           
2008-01-02  146.990005  86.940002  143.880005  86.989998  146.529999  564700   
2008-01-03  145.490005  87.309998  144.070007  87.500000  144.910004  198600   
2008-01-04  143.440002  87.949997  140.910004  88.000000  143.339996  368700   
2008-01-07  142.229996  87.870003  140.100006  87.870003  141.809998  341100   
2008-01-08  142.899994  87.870003  138.440002  88.089996  142.080002  293200   

Price                  
Ticker            SPY  
Date                   
2008-01-02  204935600  
2008-01-03  125133300  
2008-01-04  232330900  
2008-01-07  234991000  
2008-01-08  326365700

In [23]:
def select_adjusted_close(response, tickers):
    if response is None or response.empty:
        raise ValueError("Il download non ha restituito dati.")

    if "Adj Close" not in response.columns.get_level_values(0):
        raise ValueError("Il campo Adj Close manca nella risposta.")

    adjusted = response["Adj Close"]

    missing_tickers = [
        ticker for ticker in tickers
        if ticker not in adjusted.columns or adjusted[ticker].isna().all()
    ]

    if missing_tickers:
        raise ValueError(f"Nessun prezzo disponibile per: {missing_tickers}")

    return adjusted.loc[:, tickers].copy()

pilot_prices = select_adjusted_close(pilot_data, pilot_tickers)

display(pilot_prices.head())
print("Righe e colonne:", pilot_prices.shape)

Ticker,SPY,IEF
Date,,
2008-01-02,102.973946,55.000565
2008-01-03,102.924202,55.113503
2008-01-04,100.401901,55.257904
2008-01-07,100.316666,55.358276
2008-01-08,98.696739,55.477562


Righe e colonne: (2518, 2)


In [24]:
TICKERS = [
    "SPY", "QQQ", "IWM", "EFA", "EEM", "VNQ",
    "TLT", "IEF", "LQD", "HYG", "GLD", "DBC",
]

market_data = download_daily_data(TICKERS)
downloaded_at = pd.Timestamp.now(tz="UTC").isoformat()

prices = select_adjusted_close(market_data, TICKERS)

if prices.index.min() < pd.Timestamp(START_DATE):
    raise ValueError("Il file contiene date precedenti all'inizio richiesto.")

if prices.index.max() >= pd.Timestamp(END_DATE):
    raise ValueError("Il file contiene date del periodo di valutazione.")

print("Numero di righe:", len(prices))
print("Numero di ETF:", prices.shape[1])
print("Prima data:", prices.index.min().date())
print("Ultima data:", prices.index.max().date())
print("Ticker:", prices.columns.tolist())

display(prices.head())

Numero di righe: 2518
Numero di ETF: 12
Prima data: 2008-01-02
Ultima data: 2017-12-29
Ticker: ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'VNQ', 'TLT', 'IEF', 'LQD', 'HYG', 'GLD', 'DBC']


Ticker,SPY,QQQ,IWM,EFA,EEM,VNQ,TLT,IEF,LQD,HYG,GLD,DBC
Date,,,,,,,,,,,,
2008-01-02,102.973946,43.264091,58.372086,44.838005,33.408066,28.255737,52.945423,55.000565,51.003300,31.048357,84.860001,26.991880
2008-01-03,102.924202,43.444317,57.727505,44.912518,33.708637,27.374914,52.872509,55.113503,50.835007,30.989080,85.570000,27.199957
2008-01-04,100.401901,41.539017,55.987831,43.892159,32.696136,26.263496,52.883717,55.257904,51.253307,30.954746,85.129997,27.041822
2008-01-07,100.316666,41.341625,56.112091,43.938015,32.935715,26.507910,53.113667,55.358276,51.190773,30.770628,84.769997,26.550756
2008-01-08,98.696739,40.268837,54.380177,43.691528,32.680325,25.590191,53.052025,55.477562,51.094620,30.592787,86.779999,26.792126


In [25]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
raw_dir = Path("/content/drive/MyDrive/quant-research-lab/data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

prices_path = raw_dir / "etf_adj_close_2008_2017.parquet"
metadata_path = raw_dir / "etf_adj_close_2008_2017_metadata.json"

prices.to_parquet(prices_path, engine="pyarrow", index=True)

metadata = {
    "source": "Yahoo Finance via yfinance",
    "price_field": "Adj Close",
    "interval": "1d",
    "auto_adjust": False,
    "start_inclusive": START_DATE,
    "end_exclusive": END_DATE,
    "tickers": TICKERS,
    "downloaded_at_utc": downloaded_at,
    "rows": len(prices),
    "first_observation": str(prices.index.min().date()),
    "last_observation": str(prices.index.max().date()),
    "yfinance_version": yf.__version__,
    "pandas_version": pd.__version__,
    "pyarrow_version": pyarrow.__version__,
}

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Prezzi salvati in", prices_path)
print("Metadati salvati in", metadata_path)

Prezzi salvati in /content/drive/MyDrive/quant-research-lab/data/raw/etf_adj_close_2008_2017.parquet
Metadati salvati in /content/drive/MyDrive/quant-research-lab/data/raw/etf_adj_close_2008_2017_metadata.json


In [27]:
reloaded_prices = pd.read_parquet(prices_path, engine="pyarrow")

pd.testing.assert_frame_equal(reloaded_prices, prices, check_exact=True)

print("Verifica superata: il file riaperto coincide con la tabella salvata.")
print("Dimensioni:", reloaded_prices.shape)

Verifica superata: il file riaperto coincide con la tabella salvata.
Dimensioni: (2518, 12)


## Days 3–4 — Data quality and portfolio construction

In [28]:
%pip install -q pandas_market_calendars pyarrow

In [29]:
from pathlib import Path

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/quant-research-lab")
RAW_PATH = PROJECT_DIR / "data/raw/etf_adj_close_2008_2017.parquet"

START_DATE = "2008-01-01"
LAST_DATE = "2017-12-31"
EVALUATION_START = pd.Timestamp("2018-01-01")

TICKERS = [
    "SPY", "QQQ", "IWM", "EFA", "EEM", "VNQ",
    "TLT", "IEF", "LQD", "HYG", "GLD", "DBC",
]

def require_pre_evaluation(dates):
    if (dates >= EVALUATION_START).any():
        raise ValueError("The dataset contains dates from the evaluation period.")

def load_dev_data(path):
    data = pd.read_parquet(path, engine="pyarrow")

    if data.empty or not isinstance(data.index, pd.DatetimeIndex):
        raise ValueError("Expected a non-empty dataset with dates as the index.")

    if data.index.hasnans or data.index.has_duplicates:
        raise ValueError("The date index contains missing or duplicate dates.")

    require_pre_evaluation(data.index)

    if data.index.min() < pd.Timestamp(START_DATE):
        raise ValueError("The dataset contains dates before the warm-up period.")

    if data.columns.has_duplicates or set(data.columns) != set(TICKERS):
        raise ValueError("The dataset must contain exactly the 12 expected ETFs.")

    return data.loc[:, TICKERS].sort_index()

raw_prices = load_dev_data(RAW_PATH)

print("Shape:", raw_prices.shape)
print("First date:", raw_prices.index.min().date())
print("Last date:", raw_prices.index.max().date())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (2518, 12)
First date: 2008-01-02
Last date: 2017-12-29


In [30]:
test_dates = pd.to_datetime(["2017-12-29", "2018-01-02"])

try:
    require_pre_evaluation(test_dates)
except ValueError as error:
    print("Cutoff check passed:", error)
else:
    raise AssertionError("The cutoff failed to reject a 2018 date.")

Cutoff check passed: The dataset contains dates from the evaluation period.


In [31]:
nyse = mcal.get_calendar("NYSE")

expected_sessions = nyse.valid_days(
    start_date=START_DATE,
    end_date=LAST_DATE,
).tz_localize(None)

missing_sessions = expected_sessions.difference(raw_prices.index)
unexpected_sessions = raw_prices.index.difference(expected_sessions)

print("Expected sessions:", len(expected_sessions))
print("Dates in the downloaded file:", len(raw_prices))
print("Expected sessions missing from the file:", missing_sessions.tolist())
print("Dates outside the reference calendar:", unexpected_sessions.tolist())

if len(unexpected_sessions) > 0:
    raise ValueError("Review the dates outside the calendar before aligning prices.")

aligned_prices = raw_prices.reindex(expected_sessions)
aligned_prices.index.name = "Date"

Expected sessions: 2518
Dates in the downloaded file: 2518
Expected sessions missing from the file: []
Dates outside the reference calendar: []


In [32]:
invalid_prices = aligned_prices.notna() & (
    ~np.isfinite(aligned_prices) | aligned_prices.le(0)
)

usable_prices = aligned_prices.mask(invalid_prices)
asset_returns = usable_prices.pct_change(fill_method=None)

large_moves = asset_returns.abs().gt(0.15)
stale_flags = asset_returns.eq(0).rolling(5, min_periods=5).sum().eq(5)

quality_summary = pd.DataFrame({
    "missing_prices": aligned_prices.isna().sum(),
    "invalid_prices": invalid_prices.sum(),
    "large_move_flags": large_moves.sum(),
    "stale_session_flags": stale_flags.sum(),
    "valid_daily_returns": asset_returns.count(),
})

display(quality_summary)

if large_moves.any().any():
    print("Flagged daily returns, in decimal form:")
    display(asset_returns.where(large_moves).dropna(how="all"))

if invalid_prices.any().any():
    print("Zero, negative, or non-finite prices:")
    display(aligned_prices.where(invalid_prices).dropna(how="all"))

dates_with_missing_prices = aligned_prices.index[
    aligned_prices.isna().any(axis=1)
]
print("Sessions with at least one missing price:", dates_with_missing_prices.tolist())

,missing_prices,invalid_prices,large_move_flags,stale_session_flags,valid_daily_returns
Ticker,,,,,
SPY,0,0,0,0,2517
QQQ,0,0,0,0,2517
IWM,0,0,0,0,2517
EFA,0,0,1,0,2517
EEM,0,0,3,0,2517
VNQ,0,0,4,0,2517
TLT,0,0,0,0,2517
IEF,0,0,0,0,2517
LQD,0,0,0,0,2517


Flagged daily returns, in decimal form:


Ticker,SPY,QQQ,IWM,EFA,EEM,VNQ,TLT,IEF,LQD,HYG,GLD,DBC
Date,,,,,,,,,,,,
2008-10-13,NaN,NaN,NaN,0.158876,0.227699,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-10-15,NaN,NaN,NaN,NaN,-0.161662,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-10-28,NaN,NaN,NaN,NaN,0.209326,0.156373,NaN,NaN,NaN,NaN,NaN,NaN
2008-11-24,NaN,NaN,NaN,NaN,NaN,0.170066,NaN,NaN,NaN,NaN,NaN,NaN
2008-12-01,NaN,NaN,NaN,NaN,NaN,-0.195137,NaN,NaN,NaN,NaN,NaN,NaN
2009-03-23,NaN,NaN,NaN,NaN,NaN,0.161739,NaN,NaN,NaN,NaN,NaN,NaN


Sessions with at least one missing price: []


In [33]:
weights_by_portfolio = {
    "P1": pd.Series(1 / len(TICKERS), index=TICKERS),
    "P2": pd.Series({"SPY": 0.60, "IEF": 0.40}),
    "P3": pd.Series({"QQQ": 0.40, "IWM": 0.30, "EEM": 0.30}),
}

def weighted_portfolio_return(returns, weights):
    if not np.isclose(weights.sum(), 1.0):
        raise ValueError("Portfolio weights must sum to 1.")

    constituents = returns.loc[:, weights.index]
    weighted_returns = constituents.mul(weights, axis="columns")

    return weighted_returns.sum(axis=1, min_count=len(weights))

portfolio_returns = pd.DataFrame({
    name: weighted_portfolio_return(asset_returns, weights)
    for name, weights in weights_by_portfolio.items()
})

portfolio_losses = -portfolio_returns

display(portfolio_returns.head())
display(portfolio_losses.head())

,P1,P2,P3
Date,,,
2008-01-02,NaN,NaN,NaN
2008-01-03,-0.001361,0.000532,0.001053
2008-01-04,-0.016071,-0.013656,-0.035594
2008-01-07,-0.000758,0.000217,0.000963
2008-01-08,-0.007901,-0.008827,-0.021966


,P1,P2,P3
Date,,,
2008-01-02,NaN,NaN,NaN
2008-01-03,0.001361,-0.000532,-0.001053
2008-01-04,0.016071,0.013656,0.035594
2008-01-07,0.000758,-0.000217,-0.000963
2008-01-08,0.007901,0.008827,0.021966


In [34]:
test_returns = pd.DataFrame({
    "SPY": [-0.02, -0.02],
    "IEF": [0.01, np.nan],
})

test_p2 = weighted_portfolio_return(test_returns, weights_by_portfolio["P2"])

assert np.isclose(test_p2.iloc[0], -0.008)
assert pd.isna(test_p2.iloc[1])

print("Checks passed: Day 1 result and missing constituent.")

check_date = portfolio_returns["P2"].first_valid_index()

if check_date is None:
    raise ValueError("P2 has no valid daily returns.")

r_spy = asset_returns.loc[check_date, "SPY"]
r_ief = asset_returns.loc[check_date, "IEF"]
manual_p2 = 0.60 * r_spy + 0.40 * r_ief

assert np.isclose(manual_p2, portfolio_returns.loc[check_date, "P2"])

print("Date:", check_date.date())
print(f"SPY return: {r_spy:.4%}")
print(f"IEF return: {r_ief:.4%}")
print(f"P2 return: {manual_p2:.4%}")
print(f"P2 loss: {portfolio_losses.loc[check_date, 'P2']:.4%}")

Checks passed: Day 1 result and missing constituent.
Date: 2008-01-03
SPY return: -0.0483%
IEF return: 0.2053%
P2 return: 0.0532%
P2 loss: -0.0532%


In [35]:
coverage = pd.DataFrame({
    "warm_up_2008_2009": portfolio_losses.loc[:"2009-12-31"].count(),
    "development_2010_2017": portfolio_losses.loc["2010":"2017"].count(),
    "missing_daily_losses": portfolio_losses.isna().sum(),
})

display(coverage)

clean_dir = PROJECT_DIR / "data/clean"
clean_dir.mkdir(parents=True, exist_ok=True)

tables_to_save = {
    "etf_returns_2008_2017.parquet": asset_returns,
    "portfolio_returns_2008_2017.parquet": portfolio_returns,
    "portfolio_losses_2008_2017.parquet": portfolio_losses,
}

for filename, table in tables_to_save.items():
    path = clean_dir / filename
    table.to_parquet(path, engine="pyarrow", index=True)

    reloaded = pd.read_parquet(path, engine="pyarrow")
    pd.testing.assert_frame_equal(
        reloaded, table, check_exact=True, check_freq=False
    )
    print("Saved and verified:", filename)

quality_summary.to_csv(clean_dir / "data_quality_2008_2017.csv")

,warm_up_2008_2009,development_2010_2017,missing_daily_losses
P1,504,2013,1
P2,504,2013,1
P3,504,2013,1


Saved and verified: etf_returns_2008_2017.parquet
Saved and verified: portfolio_returns_2008_2017.parquet
Saved and verified: portfolio_losses_2008_2017.parquet


### Data quality and portfolio construction

Prices were aligned to a common US equity trading calendar. A daily
return requires valid prices on two consecutive trading sessions.
A portfolio return is calculated only when every constituent return
is available. Portfolio weights are reset at the start of each session.
Large moves and unchanged-price sequences are flagged for review.